# Import libraries

In [ ]:
import uuid
import copy
from datetime import datetime
from pathlib import Path
import getpass
import copy
import pandas as pd
import numpy as np
import json
import pickle
import re
import os
import glob
from typing import Dict, List, Optional, Any, Tuple
from tqdm.notebook import tqdm

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt
import seaborn as sns

pd.options.display.max_colwidth = 300

# Define project variables

In [ ]:
# Project setup
DATA_DIR = Path("/global/homes/b/bkieft/metatlas2/data/")
DATABASES_DIR = DATA_DIR / "databases"
PROJECT = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"
PROJECT_DIR = DATA_DIR / "tests/projects" / PROJECT

# QC Files Configuration
QC_FILE_PATH = PROJECT_DIR / "lcmsruns"  # Base path to search for QC files
MIN_QC_FILES = 1  # Minimum number of QC files required for RT correction

# QC Atlas Configuration (reference atlas that defines which compounds to search for)
QC_ATLAS_FILE_PATH = DATABASES_DIR / "atlases/qc-atlas-HILIC-positive-2f8ff361a0f547b6.json"  # Path to QC Atlas JSON file

# Target Atlas Configuration (atlas that will be corrected)
TARGET_ATLAS_FILE_PATH = DATABASES_DIR / "atlases/istd-atlas-HILIC-positive-c2eb0f0ffef24fc2.json"  # Path to target Atlas JSON file

# RT Corrected Atlas Data
RT_CORRECTED_ATLAS_DATA_PATH = PROJECT_DIR / "mz_rt_experimental/"
RT_CORRECTION_DATA_PATH = PROJECT_DIR / "rt_corrections/"

# Compounds and MZ/RT References Data
COMPOUNDS_DATA_PATH = DATABASES_DIR / "compounds/Compounds.json"
MZ_RT_REFERENCES_DATA_PATH = DATABASES_DIR / "mz_rt_references/MZ_RT_References.json"

# Mass Tolerance Settings
DEFAULT_MZ_TOLERANCE_PPM = 5.0  # Default m/z tolerance in ppm for compound matching
USE_ATLAS_MZ_TOLERANCE = True   # Use mz_tolerance from Atlas compounds if available

# Retention Time Settings
RT_WINDOW_EXPANSION = 1  # Extra RT window (minutes) around Atlas RT for QC peak search
MIN_PEAK_INTENSITY = 1e5  # Minimum peak intensity for QC peak consideration

# Polynomial Model Settings - ALWAYS USE SECOND-ORDER POLYNOMIAL
POLYNOMIAL_DEGREE = 2  # Fixed second-order polynomial (quadratic) for RT correction
MIN_OBSERVATIONS_PER_COMPOUND = 1 # Minimum observations (files) per compound to be included in RT correction model
MIN_COMPOUNDS_FOR_MODELING = 2  # Minimum compounds needed to build RT correction model
CROSS_VALIDATION_FOLDS = 5  # Number of folds for cross-validation

# Model Selection Criteria
R2_THRESHOLD = 0.7  # Minimum R-squared for acceptable model performance
MAX_RESIDUAL_RT = 1  # Maximum acceptable residual RT difference (minutes)

# Output Configuration
TIMESTAMP = datetime.now().isoformat()  # Timestamp for output files
GENERATE_PLOTS = True  # Generate diagnostic plots
SAVE_INTERMEDIATE_DATA = True  # Save intermediate data for debugging
ANALYST_NAME = getpass.getuser()

# Define helper functions

In [ ]:
######### Loading tools #########

def load_json_file(file_path):
    """Load a JSON file and return its content."""
    with open(file_path, 'r') as f:
        return json.load(f)

def get_project_files(project_path: str, chromatography: str, polarity: str) -> dict:
    """
    Retrieve h5 files from a specified raw data path, grouped by analysis type.

    Returns:
        dict: {'experimental': [...], 'qc': [...], 'injbl': [...], 'exctrl': [...], 'istd': [...]}
    """
    all_files = glob.glob(os.path.join(project_path, "*.h5"))

    if chromatography == 'C18':
        chromatography_key = 'C18'
    elif chromatography == 'HILIC':
        chromatography_key = 'HILICZ'
    else:
        chromatography_key = chromatography

    files_by_analysis = {'experimental': [], 'qc': [], 'injbl': [], 'exctrl': [], 'istd': []}

    abnormal_filenames = 0
    for file in all_files:
        base = os.path.basename(file)
        parts = base.split('_')

        # Check for minimum length to avoid index errors
        if len(parts) != 16:
            abnormal_filenames += 1

        # Chromatography and polarity filter
        chrom_match = parts[7].lower() == chromatography_key.lower()
        pol_match = (parts[9].lower() == polarity[:3].lower()) or (parts[9].lower() == 'fps')

        if not (chrom_match and pol_match):
            continue

        # ISTD detection
        if "-istd" in parts[14].lower():
            files_by_analysis['istd'].append(file)
            continue

        # InjBl detection
        if "-injbl" in parts[14].lower():
            files_by_analysis['injbl'].append(file)
            continue

        # Exctrl detection
        if "-exctrl" in parts[14].lower():
            files_by_analysis['exctrl'].append(file)
            continue

        # Exctrl detection
        if "-qc" in parts[14].lower():
            files_by_analysis['qc'].append(file)
            continue
        
        # Put the rest in experimental
        files_by_analysis['experimental'].append(file)

    if abnormal_filenames > 0:
        print(f"Warning: {abnormal_filenames} files have abnormal filenames and may have not been processed correctly.")

    return files_by_analysis
    
def save_project_files_db(project_path: str, project_name: str, db_path: str, chromatography: str, polarity: str, overwrite_db_entry: bool = False):
    """
    Scan the project directory for LCMS run files, group them, and save to a JSON database.

    Args:
        project_path (str): Path to the directory containing raw data files.
        project_name (str): Name of the project to be used as a key in the database.
        db_path (str): Path to the directory to save the files database.
        chromatography (str): Chromatography type ('C18' or 'HILIC').
        polarity (str): Polarity type ('positive', 'negative').
        include_istds (bool, optional): Whether to include ISTD files. Defaults to True.
        overwrite_db_entry (bool, optional): Overwrite existing entry for this project. Defaults to False.
    """
    if not os.path.exists(project_path):
        raise FileNotFoundError(f"Project path '{project_path}' does not exist.")
    os.makedirs(db_path, exist_ok=True)
    
    files_by_analysis = get_project_files(project_path, chromatography, polarity)

    full_save_path = os.path.join(db_path, 'LCMS_Runs.json')
    if os.path.exists(full_save_path):
        with open(full_save_path, 'r') as f:
            try:
                existing_data = json.load(f)
                print(f"Loaded existing LCMS runs database from {full_save_path}")
            except Exception:
                existing_data = {}
    else:
        print(f"No existing LCMS runs database found at {full_save_path}. A new one will be created.")
        existing_data = {}

    if project_name in existing_data and not overwrite_db_entry:
        print(f"LCMS run files for project '{project_name}' already exist in the database. Skipping save.")
        return files_by_analysis

    existing_data[project_name] = files_by_analysis

    with open(full_save_path, 'w') as f:
        json.dump(existing_data, f, indent=4)
    print(f"Project files database saved to {full_save_path}")

    return files_by_analysis

def expand_atlas_compounds(atlas_data, compounds_db, mz_rt_refs_data):
    """Extract and expand compounds from the Target Atlas."""

    target_compounds_data_list = []
    for compound_entry in atlas_data['compounds']:
        compound_uid = compound_entry['compound_uid']
        mz_rt_reference_uid = compound_entry['mz_rt_reference_uid']
        
        # Get compound information from compounds database (use same as QC Atlas)
        compound_info = compounds_db.get(compound_uid, {})
        compound_name = compound_info.get('name', f'Unknown_{compound_uid}')
        
        # Get combined RT and MZ information from MZ_RT references
        mz_rt_info = mz_rt_refs_data.get(mz_rt_reference_uid, {})
        rt_peak = mz_rt_info.get('rt_peak')
        rt_min = mz_rt_info.get('rt_min')
        rt_max = mz_rt_info.get('rt_max')
        mz = mz_rt_info.get('mz')
        mz_tolerance = mz_rt_info.get('mz_tolerance')
        adduct = mz_rt_info.get('adduct', 'unknown')
        
        target_compounds_data_list.append({
            'compound_uid': compound_uid,
            'compound_name': compound_name,
            'mz_rt_reference_uid': mz_rt_reference_uid,
            'rt_peak': rt_peak,
            'rt_min': rt_min,
            'rt_max': rt_max,
            'mz': mz,
            'mz_tolerance': mz_tolerance,
            'adduct': adduct
        })

    # Convert to DataFrame for easier analysis
    return pd.DataFrame(target_compounds_data_list)

def read_hdf_file(filename,desired_key=None):
    """
    Inputs:
    filename: hdf filename from which to extract desired key
    desired_key: optional key, typically "ms1_pos", "ms2_neg", etc.

    Outputs:
    df_container: a dataframe holding the information for the desired key (e.g., m/z, rt, intensity)
    """
    if desired_key is not None:

        return pd.read_hdf(filename,desired_key)

######### Analysis tools #########

def extract_ms1_data_from_file(file_path: str, polarity: str = 'positive') -> pd.DataFrame:
    """
    Extract MS1 data from a single QC file.
    
    Args:
        file_path: Path to the h5 file
        polarity: MS polarity ('positive' or 'negative')
    
    Returns:
        DataFrame with columns: mz, rt, intensity, file_path
    """
    try:
        # Determine the correct key based on polarity
        ms1_key = f"ms1_{'pos' if polarity == 'positive' else 'neg'}"
        
        # Extract MS1 data using feature_tools
        ms1_data = read_hdf_file(file_path, desired_key=ms1_key)

        if ms1_data is None or len(ms1_data) == 0:
            print(f"   No MS1 data found in {os.path.basename(file_path)}")
            return pd.DataFrame()
        
        # Add file information
        ms1_data['file_path'] = file_path
        ms1_data['filename'] = os.path.basename(file_path)
        ms1_data['polarity'] = polarity
        
        # Filter by minimum intensity
        if 'i' in ms1_data.columns:
            ms1_data = ms1_data[ms1_data['i'] >= MIN_PEAK_INTENSITY]

        rt_range_min, rt_range_max = 0.0, round(ms1_data.rt.max(), 2)

        if 'rt' in ms1_data.columns:
            ms1_data = ms1_data[ms1_data['rt'] >= rt_range_min]
            ms1_data = ms1_data[ms1_data['rt'] <= rt_range_max]

        print(f"    Extracted {len(ms1_data)} MS1 peaks from {os.path.basename(file_path)}")
        return ms1_data
        
    except Exception as e:
        print(f"    Error extracting from {os.path.basename(file_path)}: {e}")
        return pd.DataFrame()

def determine_polarity_from_atlas(atlas_method: dict) -> str:
    """Determine polarity from Atlas method information."""
    polarity = atlas_method.get('polarity', 'positive').lower()
    return 'positive' if polarity in ['positive', 'pos'] else 'negative'

def calculate_mz_tolerance_range(mz: float, tolerance_ppm: float) -> Tuple[float, float]:
    """Calculate m/z tolerance range in Daltons."""
    tolerance_da = mz * tolerance_ppm / 1e6
    return mz - tolerance_da, mz + tolerance_da

def find_peaks_in_rt_window(ms1_data: pd.DataFrame, target_mz: float, 
                           mz_tolerance_ppm: float, rt_center: float, 
                           rt_window: float = 0.5) -> pd.DataFrame:
    """
    Find peaks within m/z tolerance and RT window.
    
    Args:
        ms1_data: MS1 data DataFrame
        target_mz: Target m/z value
        mz_tolerance_ppm: m/z tolerance in ppm
        rt_center: Center RT value (minutes)
        rt_window: RT window around center (minutes)
    
    Returns:
        DataFrame of matching peaks
    """
    # Calculate m/z range
    mz_min, mz_max = calculate_mz_tolerance_range(target_mz, mz_tolerance_ppm)
    
    # Calculate RT range
    rt_min = rt_center - rt_window
    rt_max = rt_center + rt_window
    
    # Filter peaks
    matching_peaks = ms1_data[
        (ms1_data['mz'] >= mz_min) & 
        (ms1_data['mz'] <= mz_max) & 
        (ms1_data['rt'] >= rt_min) & 
        (ms1_data['rt'] <= rt_max)
    ].copy()
    
    if len(matching_peaks) > 0:
        # Calculate m/z error
        matching_peaks['mz_error_ppm'] = (
            (matching_peaks['mz'] - target_mz) / target_mz * 1e6
        )
        # Calculate RT difference
        rt_diff = matching_peaks['rt'] - rt_center
        if abs(rt_diff).any() > 1:
            print(f"RT difference exceeds threshold for {len(matching_peaks)} peaks")
        matching_peaks['rt_difference'] = rt_diff

    return matching_peaks

######### RT Alignment tools #########

def build_polynomial_model(X, y, degree):
    """Build polynomial regression model."""
    poly_features = PolynomialFeatures(degree=degree, include_bias=True)
    X_poly = poly_features.fit_transform(X.reshape(-1, 1))
    
    model = LinearRegression()
    model.fit(X_poly, y)
    
    # Calculate predictions and metrics
    y_pred = model.predict(X_poly)
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    
    return {
        'model': model,
        'poly_features': poly_features,
        'degree': degree,
        'r2': r2,
        'rmse': rmse,
        'y_pred': y_pred,
        'coefficients': model.coef_,
        'intercept': model.intercept_
    }

def predict_rt_correction(atlas_rt_values, model_info):
    """Apply RT correction model to Atlas RT values."""
    X_new = np.array(atlas_rt_values).reshape(-1, 1)
    X_new_poly = model_info['poly_features'].transform(X_new)
    corrected_rt = model_info['model'].predict(X_new_poly)
    return corrected_rt

def format_polynomial_equation(model_info):
    """Format polynomial equation as string."""
    degree = model_info['degree']
    coeffs = model_info['coefficients']
    intercept = model_info['intercept']
    
    if degree == 1:
        return f"RT_corrected = {intercept:.6f} + {coeffs[1]:.6f} * RT_atlas"
    elif degree == 2:
        return f"RT_corrected = {intercept:.6f} + {coeffs[1]:.6f} * RT_atlas + {coeffs[2]:.6f} * RT_atlas^2"
    elif degree == 3:
        return f"RT_corrected = {intercept:.6f} + {coeffs[1]:.6f} * RT_atlas + {coeffs[2]:.6f} * RT_atlas^2 + {coeffs[3]:.6f} * RT_atlas³"
    else:
        return f"Polynomial degree {degree} (coefficients: {coeffs})"


def visualize_RT_model(modeling_results_df: pd.DataFrame, best_model: dict, output_dir: str, rtc_atlas_name: str, save_plot: bool = True):

    # Sort by Atlas RT before numbering and plotting
    modeling_results_df = modeling_results_df.sort_values('ref_rt_peak').reset_index(drop=True)
    modeling_results_df['compound_num'] = modeling_results_df.index + 1

    fig = plt.figure(constrained_layout=True, figsize=(14, 10))
    gs = fig.add_gridspec(2, 2, height_ratios=[3, 1])

    # Plot 1: Atlas RT vs Observed RT (with model fit)
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.scatter(modeling_results_df['ref_rt_peak'], modeling_results_df['exp_rt_median'], 
                alpha=0.7, s=50, c='blue', label='Observed Data')
    ax1.plot(modeling_results_df['ref_rt_peak'], modeling_results_df['predicted_rt'], 
            'r-', linewidth=2, label='Polynomial Fit (degree 2)')
    ax1.plot([modeling_results_df['ref_rt_peak'].min(), modeling_results_df['ref_rt_peak'].max()],
            [modeling_results_df['ref_rt_peak'].min(), modeling_results_df['ref_rt_peak'].max()],
            'k--', alpha=0.5, label='Perfect Correlation')
    ax1.set_xlabel('Atlas RT (min)')
    ax1.set_ylabel('Observed RT (min)')
    ax1.set_title(f'RT Correlation (R² = {best_model["r2"]:.4f})')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Add numbers to points
    for _, row in modeling_results_df.iterrows():
        ax1.annotate(str(int(row['compound_num'])), 
                    (row['ref_rt_peak'], row['exp_rt_median']),
                    textcoords="offset points", xytext=(5, -10), ha='left', fontsize=9, color='black')

    # Plot 2: Residuals vs Atlas RT
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.scatter(modeling_results_df['ref_rt_peak'], modeling_results_df['residual'], 
                alpha=0.7, s=50, c='green')
    ax2.axhline(y=0, color='red', linestyle='--', alpha=0.7)
    ax2.axhline(y=modeling_results_df['residual'].std(), color='orange', linestyle=':', alpha=0.7, label='+1 σ')
    ax2.axhline(y=-modeling_results_df['residual'].std(), color='orange', linestyle=':', alpha=0.7, label='-1 σ')
    ax2.set_xlabel('Atlas RT (min)')
    ax2.set_ylabel('Residual (min)')
    ax2.set_title(f'Residuals vs Atlas RT (RMSE = {best_model["rmse"]:.4f})')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # Add numbers to points
    for _, row in modeling_results_df.iterrows():
        ax2.annotate(str(int(row['compound_num'])), 
                    (row['ref_rt_peak'], row['residual']),
                    textcoords="offset points", xytext=(5, -10), ha='left', fontsize=9, color='black')

    # Table: Compound number to name mapping (ordered by RT), now with residuals
    ax_table = fig.add_subplot(gs[1, :])
    ax_table.axis('off')
    table_data = modeling_results_df[['compound_num', 'compound_name', 'inchi_key', 'ref_rt_peak', 'residual']].copy()
    table_data['ref_rt_peak'] = table_data['ref_rt_peak'].round(3)
    table_data['residual'] = table_data['residual'].round(4)
    table_data.columns = ['#', 'Compound Name', 'InChi Key', 'Atlas RT (min)', 'Residual (min)']
    table = ax_table.table(cellText=table_data.values,
                        colLabels=table_data.columns,
                        loc='center',
                        cellLoc='left',
                        colLoc='left')

    # Set column widths as requested: 0.1, 0.3, 0.3, 0.15, 0.15
    col_widths = [0.2, 0.5, 0.5, 0.3, 0.3]
    for i, width in enumerate(col_widths):
        table.auto_set_column_width(i)
        for j in range(len(table_data) + 1):  # +1 for header
            cell = table[(j, i)]
            cell.set_width(width)

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.2)

    plt.suptitle('RT Correction Model Validation', fontsize=16, fontweight='bold', y=1.02)

    if save_plot:
        # Save the plot as PDF
        output_dir.mkdir(parents=True, exist_ok=True)
        pdf_path = output_dir / f"summary-for-{rtc_atlas_name}.pdf"
        plt.savefig(pdf_path, bbox_inches='tight')
        plt.show()
        print(f"Plot saved to {pdf_path}")


def create_RT_summary(compounds: pd.DataFrame, best_model: dict, qc_files: list, target_compounds: pd.DataFrame, modeling_data: list, 
                      output_dir: str, rtc_atlas_name: str, save_summary: bool = True):

    corrected_mz_rt_experimental_data = {}
    correction_stats = []

    for _, row in compounds.iterrows():
        entry = row.to_dict()
        if entry['rt_peak'] is not None:
            new_uid = f"mzrt-exp-{uuid.uuid4().hex[:16]}"
            corrected_rt = predict_rt_correction([entry['rt_peak']], best_model)[0]
            window = (entry['rt_max'] - entry['rt_min']) if entry['rt_min'] is not None and entry['rt_max'] is not None else None
            corrected_min = corrected_rt - window / 2 if window is not None else None
            corrected_max = corrected_rt + window / 2 if window is not None else None
            rt_shift = corrected_rt - entry['rt_peak']
            corrected_mz_rt_experimental_data[new_uid] = {
                "mz_rt_experimental_uid": new_uid,
                "compound_uid": entry['compound_uid'],
                "rt_peak": corrected_rt,
                "rt_min": corrected_min,
                "rt_max": corrected_max,
                "rt_units": "min",
                "mz": entry['mz'],
                "mz_tolerance": entry['mz_tolerance'],
                "mz_tolerance_units": "ppm",
                "adduct": entry['adduct'],
                "charge": 1 if '+' in str(entry['adduct']) else -1 if '-' in str(entry['adduct']) else 0,
                "last_modified": TIMESTAMP,
                "updated_from_ref": True,
                "data_source": "rt_correction",
                "source_mz_rt_reference_uid": entry['mz_rt_reference_uid'],
                "rt_correction_metadata": {
                    "correction_applied": True,
                    "rt_shift": rt_shift,
                    "model_degree": best_model['degree'],
                    "model_r2": best_model['r2'],
                    "model_rmse": best_model['rmse'],
                    "correction_timestamp": TIMESTAMP,
                    "correction_method": "polynomial_qc_based",
                    "source_qc_atlas": os.path.basename(QC_ATLAS_FILE_PATH),
                    "qc_files_count": len(qc_files),
                    "qc_compounds_used_for_modeling": len(modeling_data),
                    "model_equation": format_polynomial_equation(best_model)
                }
            }
            correction_stats.append({
                'compound_name': entry['compound_name'],
                'compound_uid': entry['compound_uid'],
                'mz_rt_reference_uid': entry['mz_rt_reference_uid'],
                'mz_rt_experimental_uid': new_uid,
                'original_rt': entry['rt_peak'],
                'corrected_rt': corrected_rt,
                'rt_shift': rt_shift
            })
        else:
            print(f"Warning! Original RT peak is None, skipping correction for compound: {entry['compound_name']}")

    correction_df = pd.DataFrame(correction_stats)
    summary = {
        'total_compounds': len(target_compounds),
        'corrected_compounds': len(correction_stats),
        'uncorrected_compounds': len(target_compounds) - len(correction_stats),
        'correction_stats': correction_stats,
        'mean_correction': correction_df['rt_shift'].mean() if not correction_df.empty else 0,
        'std_correction': correction_df['rt_shift'].std() if not correction_df.empty else 0,
        'min_correction': correction_df['rt_shift'].min() if not correction_df.empty else 0,
        'max_correction': correction_df['rt_shift'].max() if not correction_df.empty else 0
    }

    print(f"RT correction completed: {summary['corrected_compounds']}/{summary['total_compounds']} compounds corrected")
    if summary['corrected_compounds']:
        print(f"Correction statistics: mean = {summary['mean_correction']:.4f}, std = {summary['std_correction']:.4f} min")

    if save_summary:
        output_dir.mkdir(parents=True, exist_ok=True)
        alignment_summary_file = output_dir / f"summary-for-{rtc_atlas_name}.json"
        with open(alignment_summary_file, 'w') as f:
            json.dump(summary, f, indent=2)
        print(f"RT alignment summary saved to {alignment_summary_file}")

# Load Compounds and MZ/RT References Databases

In [ ]:
# Load referenced MZ_RT data files
mz_rt_references_data = load_json_file(MZ_RT_REFERENCES_DATA_PATH)
print(f"MZ_RT references loaded: {len(mz_rt_references_data)} entries")

# Load compounds database if available
compounds_db = load_json_file(COMPOUNDS_DATA_PATH)
print(f"Compounds database loaded: {len(compounds_db)} compounds")

# Load QC Atlas

In [ ]:
# Load the QC Atlas JSON file
qc_atlas_data = load_json_file(QC_ATLAS_FILE_PATH)
print(f"QC Atlas loaded from {QC_ATLAS_FILE_PATH}")

# Extract method information for polarity
qc_atlas_method = qc_atlas_data.get('method', {})

# Expand compounds from the QC Atlas
qc_compounds_df = expand_atlas_compounds(qc_atlas_data, compounds_db, mz_rt_references_data)

# Filter out compounds without RT or MZ data
valid_qc_compounds_df = qc_compounds_df.dropna(subset=['rt_peak', 'mz'])

# Check load
if len(valid_qc_compounds_df) == 0:
    print("No valid compounds found in QC Atlas. Exiting.")
    exit(1)

# Show basic Atlas information
display(pd.DataFrame({
    'Atlas UID': [qc_atlas_data['atlas_uid']],
    'Atlas Name': [qc_atlas_data['atlas_name']],
    'Compounds': [len(valid_qc_compounds_df)],
    'Polarity': [qc_atlas_method.get('polarity', 'unknown')],
    'Chromatography': [qc_atlas_method.get('chromatography', 'unknown')]
}))

# Load Target Atlas

In [ ]:
# Load the Target Atlas JSON file
target_atlas_data = load_json_file(TARGET_ATLAS_FILE_PATH)
print(f"Target Atlas loaded from {TARGET_ATLAS_FILE_PATH}")

# Extract method information for polarity
target_atlas_method = target_atlas_data.get('method', {})

# Expand compounds from the QC Atlas
target_compounds_df = expand_atlas_compounds(target_atlas_data, compounds_db, mz_rt_references_data)

# Filter out compounds without RT or MZ data
valid_target_compounds_df = target_compounds_df.dropna(subset=['rt_peak', 'mz'])

# Check load
if len(valid_target_compounds_df) == 0:
    print("No valid compounds found in Target Atlas. Exiting.")
    exit(1)

# Show basic Atlas information
display(pd.DataFrame({
    'Atlas UID': [target_atlas_data['atlas_uid']],
    'Atlas Name': [target_atlas_data['atlas_name']],
    'Compounds': [len(valid_target_compounds_df)],
    'Polarity': [target_atlas_method.get('polarity', 'unknown')],
    'Chromatography': [target_atlas_method.get('chromatography', 'unknown')]
}))

# Define corrected atlas
rtc_atlas_uid = f"rtc-{target_atlas_data['atlas_uid'][:-17]}-{uuid.uuid4().hex[:16]}"

# Save LCMS runs to database

In [ ]:
lcmsrun_files = save_project_files_db(
                    project_path=PROJECT_DIR / "lcmsruns",
                    project_name=PROJECT,
                    db_path=DATA_DIR / "databases/lcmsruns/",
                    chromatography=target_atlas_method.get('chromatography', 'unknown'),
                    polarity=target_atlas_method.get('polarity', 'unknown'),
                    overwrite_db_entry=True
                )

# Load LCMS run files

In [ ]:
# Filter and validate files
valid_qc_files = []
qc_files = lcmsrun_files['qc']
for qc_file in qc_files:
    if os.path.isfile(qc_file) and os.path.getsize(qc_file) > 0:
        valid_qc_files.append(qc_file)
        print(f"  {os.path.basename(qc_file)}")
    else:
        print(f"  {os.path.basename(qc_file)} (invalid or empty)")

qc_files = valid_qc_files
print(f"\nValid QC files ready for processing: {len(qc_files)}")

if len(qc_files) < MIN_QC_FILES:
    raise ValueError(f"\nInsufficient QC files found. Need at least {MIN_QC_FILES}, but found {len(qc_files)}")

# Extract RT Data from QC files

In [ ]:
# Extract MS1 data from all QC files
print("Extracting targeted MS1 data from QC files...")

# Determine polarity from QC Atlas method information
polarity = determine_polarity_from_atlas(qc_atlas_method)

# Extract MS1 data from each QC file
all_ms1_data = []
extraction_stats = {
    'total_files': len(qc_files),
    'successful_extractions': 0,
    'failed_extractions': 0,
    'total_peaks': 0
}

for qc_file in tqdm(qc_files, desc="Extracting MS1 data"):
    ms1_data = extract_ms1_data_from_file(qc_file, polarity)
    
    if len(ms1_data) > 0:
        all_ms1_data.append(ms1_data)
        extraction_stats['successful_extractions'] += 1
        extraction_stats['total_peaks'] += len(ms1_data)
    else:
        extraction_stats['failed_extractions'] += 1

# Combine all MS1 data
if all_ms1_data:
    combined_ms1_data = pd.concat(all_ms1_data, ignore_index=True)
    print(f"\nMS1 data extraction completed")
    print(f"  Successful extractions: {extraction_stats['successful_extractions']}/{extraction_stats['total_files']}")
    print(f"  Total MS1 peaks: {extraction_stats['total_peaks']:,}")
    
    # Display data summary
    print(f"\nMS1 Data Summary:")
    print(f"  RT range: {combined_ms1_data['rt'].min():.2f} - {combined_ms1_data['rt'].max():.2f} min")
    print(f"  m/z range: {combined_ms1_data['mz'].min():.4f} - {combined_ms1_data['mz'].max():.4f}")
    print(f"  Intensity range: {combined_ms1_data['i'].min():.0f} - {combined_ms1_data['i'].max():.0f}")
    
else:
    raise ValueError("No MS1 data could be extracted from any QC files")

# Match QC Atlas Compounds to Extracted QC MS1 Data

In [ ]:
# Match QC Atlas compounds with extracted QC data
print("Matching QC Atlas compounds with QC peak data")

# Track matching statistics
compound_matches = []
matching_stats = {
    'total_compounds': len(valid_qc_compounds_df),
    'compounds_with_matches': 0,
    'compounds_without_matches': 0,
    'total_matches': 0,
    'multiple_matches': 0
}

for idx, compound in tqdm(valid_qc_compounds_df.iterrows(), 
                         total=len(valid_qc_compounds_df), 
                         desc="Matching QC compounds"):
    
    compound_name = compound['compound_name']
    target_mz = compound['mz']
    atlas_rt = compound['rt_peak']
    
    # Use compound-specific m/z tolerance if available
    if USE_ATLAS_MZ_TOLERANCE and pd.notna(compound['mz_tolerance']):
        mz_tolerance = compound['mz_tolerance']
    else:
        mz_tolerance = DEFAULT_MZ_TOLERANCE_PPM
    
    # Find matching peaks in QC data
    matching_peaks = find_peaks_in_rt_window(
        combined_ms1_data, 
        target_mz, 
        mz_tolerance, 
        atlas_rt, 
        RT_WINDOW_EXPANSION
    )
    
    if len(matching_peaks) > 0:
        matching_stats['compounds_with_matches'] += 1
        matching_stats['total_matches'] += len(matching_peaks)
        
        if len(matching_peaks) > 1:
            matching_stats['multiple_matches'] += 1
        
        # Group by file and take the most intense peak per file
        file_grouped = matching_peaks.groupby('filename')
        best_peaks_per_file = []
        
        for filename, file_peaks in file_grouped:
            # Take the peak with highest intensity
            best_peak = file_peaks.loc[file_peaks['i'].idxmax()]
            best_peaks_per_file.append({
                'compound_uid': compound['compound_uid'],
                'compound_name': compound_name,
                'atlas_rt_peak': atlas_rt,
                'atlas_rt_min': compound['rt_min'],
                'atlas_rt_max': compound['rt_max'],
                'atlas_mz': target_mz,
                'observed_rt': best_peak['rt'],
                'observed_mz': best_peak['mz'],
                'observed_intensity': best_peak['i'],
                'mz_error_ppm': best_peak['mz_error_ppm'],
                'rt_difference': best_peak['rt_difference'],
                'filename': filename,
                'file_path': best_peak['file_path'],
                'mz_tolerance_used': mz_tolerance
            })
        
        compound_matches.extend(best_peaks_per_file)
        
    else:
        matching_stats['compounds_without_matches'] += 1

# Convert to DataFrame
if compound_matches:
    matches_df = pd.DataFrame(compound_matches)
    # Add inchi_key to matches_df as column based on compound_uid
    matches_df['inchi_key'] = matches_df['compound_uid'].apply(
        lambda uid: compounds_db.get(uid, {}).get('inchi_key', 'unknown')
    )
    
    print(f"Compounds with matches: {matching_stats['compounds_with_matches']}")
    print(f"Compounds without matches: {matching_stats['compounds_without_matches']}")
    print(f"Total peak matches: {matching_stats['total_matches']}")
    
    # Summary statistics
    print(f"Mean m/z error: {matches_df['mz_error_ppm'].mean():.2f} ± {matches_df['mz_error_ppm'].std():.2f} ppm")
    print(f"Mean RT difference: {matches_df['rt_difference'].mean():.3f} ± {matches_df['rt_difference'].std():.3f} min")
    
else:
    print("No QC compound matches found. Check Atlas compound definitions, m/z tolerance, RT window settings, and QC file data quality")
    raise ValueError("No compound matches found")

# Build RT Alignment Model

In [ ]:
# Optionally filter out QC compounds by InChI Key
# Provide a list or set of InChI Keys to exclude
EXCLUDE_INCHIKEYS = ["OVRNDRQMDRJTHS-ZEUBEQSHSA-N"]  # e.g., {'ABCDEFGHIJKL-MNOPQRSTUV-Z', ...}

if EXCLUDE_INCHIKEYS:
    # Ensure 'inchi_key' column exists in matches_df
    if 'inchi_key' not in matches_df.columns:
        raise ValueError("matches_df must contain an 'inchi_key' column to filter by InChI Key.")
    before_count = len(matches_df)
    matches_df = matches_df[~matches_df['inchi_key'].isin(EXCLUDE_INCHIKEYS)]
    after_count = len(matches_df)
    print(f"Filtered out {before_count - after_count} matches by InChI Key.")

# Build RT correction model using QC compound statistics

# Aggregate matches by compound to calculate median/mean observed RT values
compound_rt_stats = matches_df.groupby(['compound_uid', 'compound_name', 'inchi_key']).agg({
    'atlas_rt_peak': 'first',
    'atlas_rt_min': 'first',
    'atlas_rt_max': 'first',
    'atlas_mz': 'first',
    'observed_rt': ['mean', 'median', 'std', 'count'],
    'observed_mz': ['mean', 'std'],
    'observed_intensity': ['mean', 'median', 'max'],
    'rt_difference': ['mean', 'median', 'std'],
    'mz_error_ppm': ['mean', 'std']
}).round(4)

# Flatten column names
compound_rt_stats.columns = [
    'ref_rt_peak',           # Atlas RT peak (reference)
    'ref_rt_min',            # Atlas RT min
    'ref_rt_max',            # Atlas RT max  
    'ref_mz',                # Atlas m/z
    'exp_rt_mean',           # Mean observed RT across files
    'exp_rt_median',         # Median observed RT across files (more robust)
    'exp_rt_std',            # Standard deviation of observed RT
    'observation_count',     # Number of files where compound was observed
    'exp_mz_mean',           # Mean observed m/z
    'exp_mz_std',            # Std dev of observed m/z
    'exp_intensity_mean',    # Mean intensity
    'exp_intensity_median',  # Median intensity  
    'exp_intensity_max',     # Max intensity
    'rt_diff_mean',          # Mean RT difference (observed - atlas)
    'rt_diff_median',        # Median RT difference
    'rt_diff_std',           # Std dev of RT difference
    'mz_error_mean',         # Mean m/z error (ppm)
    'mz_error_std'           # Std dev of m/z error (ppm)
]

# Reset index to get compound info as columns
compound_rt_stats = compound_rt_stats.reset_index()

# Display summary statistics
print(f"\nRT Statistics Summary:")
print(f"  Atlas RT range: {compound_rt_stats['ref_rt_peak'].min():.2f} - {compound_rt_stats['ref_rt_peak'].max():.2f} min")
print(f"  Observed RT range (median): {compound_rt_stats['exp_rt_median'].min():.2f} - {compound_rt_stats['exp_rt_median'].max():.2f} min")
print(f"  Mean RT difference (observed - atlas): {compound_rt_stats['rt_diff_median'].mean():.3f} ± {compound_rt_stats['rt_diff_median'].std():.3f} min")

# Show the compound stats table
print(f"\nCompound RT Statistics:")
display(compound_rt_stats[['compound_name', 'inchi_key', 'ref_rt_peak', 'exp_rt_median', 'rt_diff_median', 
                          'observation_count', 'exp_rt_std']])

# Prepare modeling data from compound RT statistics
modeling_data = []

# Filter compounds with sufficient observations for reliable modeling
reliable_compounds = compound_rt_stats[compound_rt_stats['observation_count'] >= MIN_OBSERVATIONS_PER_COMPOUND]
print(f"Using {len(reliable_compounds)} compounds with ≥{MIN_OBSERVATIONS_PER_COMPOUND} observations (QC files) for modeling")
if len(reliable_compounds) < MIN_COMPOUNDS_FOR_MODELING:
    raise ValueError(f"Insufficient compounds for modeling. Need at least {MIN_COMPOUNDS_FOR_MODELING}, but found {len(reliable_compounds)}")

# Extract X (Atlas RT) and y (observed median RT) for modeling
X_atlas_rt = reliable_compounds['ref_rt_peak'].values
y_observed_rt = reliable_compounds['exp_rt_median'].values

# Create modeling dataset
modeling_results_df = reliable_compounds.copy()

# Build second-order polynomial model (as specified in config)
print(f"Building polynomial model (degree {POLYNOMIAL_DEGREE})...")

best_model = build_polynomial_model(X_atlas_rt, y_observed_rt, POLYNOMIAL_DEGREE)

# Add predictions and residuals to modeling results
modeling_results_df['predicted_rt'] = best_model['y_pred']
modeling_results_df['residual'] = y_observed_rt - best_model['y_pred']
modeling_results_df['abs_residual'] = np.abs(modeling_results_df['residual'])

# Model evaluation
print(f"\nModel built successfully:")
print(f"  Model type: Polynomial degree {best_model['degree']}")
print(f"  R² = {best_model['r2']:.4f}")
print(f"  RMSE = {best_model['rmse']:.4f} min")
#print(f"  Cross-validation R² = {cv_info['cv_mean_r2']:.4f} ± {cv_info['cv_std_r2']:.4f}")
print(f"  Max residual = {modeling_results_df['abs_residual'].max():.4f} min")

# Display polynomial equation
equation = format_polynomial_equation(best_model)
print(f"  Equation: {equation}")
best_model['equation'] = equation

# Check model quality
if best_model['r2'] < R2_THRESHOLD:
    print(f"Warning: Model R² ({best_model['r2']:.4f}) is below threshold ({R2_THRESHOLD})")

if modeling_results_df['abs_residual'].max() > MAX_RESIDUAL_RT:
    print(f"Warning: Maximum residual ({modeling_results_df['abs_residual'].max():.4f} min) exceeds threshold ({MAX_RESIDUAL_RT} min)")

# Add compounds
best_model['compounds_used_for_modeling'] = reliable_compounds['compound_uid'].tolist()

# Store modeling data for later use
modeling_data = modeling_results_df.to_dict('records')

# Create RT-Corrected Atlas and RT Experimental Data

In [ ]:
# Create RT-Corrected Atlas and MZ_RT Experimental Data
print("Creating RT-corrected Atlas and MZ_RT Experimental data...")

# Create RT-corrected Atlas by copying Target Atlas with modifications
rtc_atlas_data = copy.deepcopy(target_atlas_data)
rtc_atlas_data.update({
    "atlas_uid": rtc_atlas_uid,
    "atlas_name": target_atlas_data['atlas_name'] + "_rtc-" + datetime.now().strftime('%Y%m%d%H%M%S'),
    "atlas_description": target_atlas_data['atlas_description'] + " (RT-corrected)",
    "creation_time": TIMESTAMP,
    "last_modified": TIMESTAMP,
    "notes": target_atlas_data['notes'] + " (RT-corrected)",
    "source_atlas_uid": target_atlas_data['atlas_uid'],
    "rtc_model_info": {
        "polynomial_degree": best_model['degree'],
        "r_squared": best_model['r2'],
        "coefficients": best_model['coefficients'].tolist(),
        "equation": best_model['equation'],
        "rmse": best_model['rmse'],
        "qc_files_used": len(qc_files),
        "compounds_used_for_modeling": len(modeling_data),
        "rt_correction_date": TIMESTAMP
    }
})

# Create combined MZ_RT Experimental data by copying MZ_RT_References with RT corrections applied
mz_rt_experimental_data = {}
updated_compounds = []
correction_stats = []

for compound in target_atlas_data['compounds']:
    compound_uid = compound['compound_uid']
    original_mz_rt_reference_uid = compound['mz_rt_reference_uid']
    
    # Look up original MZ_RT reference data
    if original_mz_rt_reference_uid not in mz_rt_references_data:
        print(f"Warning: MZ_RT reference {original_mz_rt_reference_uid} not found for compound {compound_uid}")
        continue
    
    # Get reference data to adjust
    original_mz_rt_data = mz_rt_references_data[original_mz_rt_reference_uid]

    # Generate new MZ_RT experimental UID
    mz_rt_experimental_uid = f"mzrt-exp-{uuid.uuid4().hex[:16]}"
    
    # Apply RT correction using the polynomial model
    original_rt_peak = original_mz_rt_data['rt_peak']

    # Transform the RT using polynomial features (for degree 2: [1, rt, rt^2])
    rt_poly = best_model['poly_features'].transform([[original_rt_peak]])
    corrected_rt_peak = best_model['model'].predict(rt_poly)[0]
    rt_shift = corrected_rt_peak - original_rt_peak
    
    # Calculate corrected RT window
    rt_window_size = original_mz_rt_data['rt_max'] - original_mz_rt_data['rt_min']
    corrected_rt_min = corrected_rt_peak - (rt_window_size / 2)
    corrected_rt_max = corrected_rt_peak + (rt_window_size / 2)
    
    # Create combined MZ_RT Experimental entry (combines both MZ and RT data)
    mz_rt_experimental_entry = {
        "mz_rt_experimental_uid": mz_rt_experimental_uid,
        "compound_uid": compound_uid,
        "compound_name": compound.get('compound_name', ''),
        "method_uid": original_mz_rt_data.get('method_uid', ''),

        # Original values for reference
        "mz_rt_reference_uid": original_mz_rt_reference_uid,

        # Corrected RT values
        "rt_peak": corrected_rt_peak,
        "rt_min": corrected_rt_min,
        "rt_max": corrected_rt_max,
        "rt_units": original_mz_rt_data.get('rt_units', 'min'),
        
        # MZ values (unchanged from reference)
        "mz": original_mz_rt_data['mz'],
        "mz_tolerance": original_mz_rt_data['mz_tolerance'],
        "mz_tolerance_units": original_mz_rt_data.get('mz_tolerance_units', 'ppm'),
        "adduct": original_mz_rt_data['adduct'],
        "charge": original_mz_rt_data.get('charge', 0),
        
        # Correction metadata
        "rt_correction_applied": True,
        "rt_shift": rt_shift,
        
        # Provenance
        "parent_atlas_uid": target_atlas_data['atlas_uid'],
        "rtc_atlas_uid": rtc_atlas_uid,
        "source": original_mz_rt_data['source'],
        
        # Standard reference fields
        "created_by": ANALYST_NAME,
        "creation_time": TIMESTAMP,
        "last_modified": TIMESTAMP
    }
    
    mz_rt_experimental_data[mz_rt_experimental_uid] = mz_rt_experimental_entry
    
    # Update compound with mz_rt_experimental_uid
    updated_compound = copy.deepcopy(compound)
    updated_compound["mz_rt_experimental_uid"] = mz_rt_experimental_uid

    updated_compounds.append(updated_compound)

    # Collect correction stats for summary
    correction_stats.append({
        'compound_name': compound.get('compound_name', ''),
        'compound_uid': compound_uid,
        'mz_rt_reference_uid': original_mz_rt_reference_uid,
        'mz_rt_experimental_uid': mz_rt_experimental_uid,
        'original_rt': original_rt_peak,
        'corrected_rt': corrected_rt_peak,
        'rt_shift': rt_shift
    })

# Update the RT-corrected Atlas with modified compounds
rtc_atlas_data['compounds'] = updated_compounds

# Save files
output_dir = Path(TARGET_ATLAS_FILE_PATH).parent
mz_rt_exp_output_dir = Path(RT_CORRECTED_ATLAS_DATA_PATH)
mz_rt_exp_output_dir.mkdir(parents=True, exist_ok=True)

# Update Atlas with actual file path for experimental data
mz_rt_experimental_file_path = mz_rt_exp_output_dir / f"mz-rt-exp-{rtc_atlas_uid}.json"
rtc_atlas_data['rt_mz_source'] = {
    'mz_rt_experimental': str(mz_rt_experimental_file_path)
}

# Save RT-corrected Atlas
rtc_atlas_file_path = output_dir / f"{rtc_atlas_uid}.json"
with open(rtc_atlas_file_path, 'w') as f:
    json.dump(rtc_atlas_data, f, indent=2)
print(f"   RT-corrected Atlas saved to {rtc_atlas_file_path}")

# Save MZ_RT Experimental data
with open(mz_rt_experimental_file_path, 'w') as f:
    json.dump(mz_rt_experimental_data, f, indent=2)
print(f"   MZ_RT Experimental data saved to {mz_rt_experimental_file_path}")

# Create plot summary of RT
print("Visualizing RT correction model...")
visualize_RT_model(modeling_results_df, output_dir=RT_CORRECTION_DATA_PATH, best_model=best_model, rtc_atlas_name=rtc_atlas_uid, save_plot=True)

print("Creating RT alignment summary...")
create_RT_summary(compounds=target_compounds_df, best_model=best_model, qc_files=qc_files, target_compounds=target_compounds_df, modeling_data=modeling_data, 
                  output_dir=RT_CORRECTION_DATA_PATH, rtc_atlas_name=rtc_atlas_uid, save_summary=True)
